# README
## Purpose
Train and explore a Top2Vec baseline on combined timeline summaries.
## Inputs
- `Data/combined_timeline_data.csv`
## Outputs
- Topic sizes, top topic words, and document-topic retrieval outputs.
## Notes
This notebook provides a neural baseline complementary to BERTopic and LDA experiments.

Imports

In [1]:
import pandas as pd
from top2vec import Top2Vec

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Data

In [2]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


Prepare Data

In [3]:
docs = df["Summary"].tolist()

Prepare Model

In [4]:
model = Top2Vec(docs)

2026-04-16 21:30:10,434 - top2vec - INFO - Pre-processing documents for training
C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-04-16 21:30:10,492 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model
C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-04-16 21:30:10,492 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1005.62it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:


Analyze Topics

In [5]:
topic_sizes, topic_nums = model.get_topic_sizes()
print(topic_sizes)
print(topic_nums)
topic_words, word_scores, topic_nums = model.get_topics(3)
for words, scores, num in zip(topic_words, word_scores, topic_nums):
    print(num)
    print(f"Words: {words}")

[106  95  79  56  39  37  30  28  28  22]
[0 1 2 3 4 5 6 7 8 9]
0
Words: ['banks' 'bank' 'financial' 'securities' 'treasury' 'lending' 'reserve'
 'fdic' 'loans' 'funds' 'federal' 'loan' 'institutions' 'liquidity'
 'credit' 'market' 'capital' 'stock' 'asset' 'releases' 'backed' 'board'
 'department' 'announces' 'covid' 'eligible' 'facility' 'states' 'report'
 'primary' 'rate' 'will' 'term' 'to' 'under' 'program' 'it' 'for'
 'through' 'new' 'at' 'billion' 'purchase' 'on' 'up' 'in' 'act'
 'preferred' 'that' 'also']
1
Words: ['banks' 'bank' 'financial' 'securities' 'funds' 'lending' 'loans' 'loan'
 'fdic' 'treasury' 'reserve' 'federal' 'market' 'liquidity' 'stock'
 'credit' 'asset' 'institutions' 'backed' 'facility' 'capital' 'purchase'
 'board' 'releases' 'eligible' 'rate' 'primary' 'department' 'billion'
 'announces' 'total' 'term' 'new' 'also' 'percent' 'its' 'program'
 'states' 'at' 'for' 'will' 'to' 'it' 'through' 'preferred' 'on' 'up'
 'under' 'in' 'report']
2
Words: ['covid' 'states

Analyze Documents

In [6]:
documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=0, num_docs=10)

Prepare Validation

In [7]:
# Top2Vec
top2vec_topics = [list(words) for words in model.get_topics()[0]]

In [8]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.corpora import Dictionary

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Preprocess documents for gensim
stop_words = set(stopwords.words('english'))
processed_docs = []
for doc in docs:
    tokens = word_tokenize(doc.lower())
    tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    processed_docs.append(tokens)

# Create dictionary
id2word = Dictionary(processed_docs)

In [9]:
from gensim.models.coherencemodel import CoherenceModel

def calculate_metrics(topics, corpus_docs, dictionary):
    # 1. Coherence (Cv)
    cm = CoherenceModel(topics=topics, texts=corpus_docs, dictionary=dictionary, coherence='c_v')
    coherence = cm.get_coherence()
    
    # 2. Topic Diversity
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words) if len(all_words) > 0 else 0
    
    return coherence, diversity

# Calculate metrics for Top2Vec
cv_top2vec, div_top2vec = calculate_metrics(top2vec_topics, processed_docs, id2word)
print(f"Top2Vec Coherence (C_v): {cv_top2vec:.4f}")
print(f"Top2Vec Topic Diversity: {div_top2vec:.4f}")

Top2Vec Coherence (C_v): 0.3679
Top2Vec Topic Diversity: 0.1320


## Export outputs for cross-model comparison

In [10]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np

pipeline_name = 'Top2Vec'
required = ['df', 'docs', 'model']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing required variables: {missing}. Run the full notebook first.')

run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')
base_dir = Path('Outputs') / 'Comparisons_03' / pipeline_name / run_tag
folders = {
    'tables': base_dir / 'tables',
    'row_level': base_dir / 'row_level',
    'models': base_dir / 'models',
    'meta': base_dir / 'meta'
}
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)


def save_df(df_obj, stem, folder):
    csv_path = folder / f'{stem}.csv'
    df_obj.to_csv(csv_path, index=False, encoding='utf-8-sig')
    return str(csv_path)

exported = {}
notes = []

# Topic-level tables
topic_words, topic_word_scores, topic_nums = model.get_topics()
topic_terms_rows = []
for idx, topic_id in enumerate(topic_nums):
    words = topic_words[idx]
    scores = topic_word_scores[idx]
    for rank, (term, score) in enumerate(zip(words, scores), start=1):
        topic_terms_rows.append({
            'topic_id': int(topic_id),
            'term_rank': int(rank),
            'term': str(term),
            'weight': float(score)
        })

topic_terms_df = pd.DataFrame(topic_terms_rows)
exported['topic_terms'] = save_df(topic_terms_df, 'topic_terms_long', folders['tables'])

# Row-level assignments via Top2Vec API
summary_col = 'Summary' if 'Summary' in df.columns else df.columns[0]
date_col = 'Date' if 'Date' in df.columns else None
n = min(len(docs), len(df))

try:
    doc_ids = list(range(n))
    doc_topics, doc_topic_scores, _, _ = model.get_documents_topics(doc_ids)

    row_df = df.iloc[:n].copy().reset_index(drop=True)
    assignments_df = pd.DataFrame({
        'row_index': np.arange(n),
        'text': row_df[summary_col].astype(str),
        'topic': pd.Series(doc_topics).astype('int64'),
        'topic_score': pd.Series(doc_topic_scores).astype(float)
    })
    if date_col is not None:
        assignments_df['date'] = row_df[date_col].astype(str)

    exported['row_topic_assignments'] = save_df(assignments_df, 'row_topic_assignments', folders['row_level'])

    topic_prevalence_df = (
        assignments_df.groupby('topic', as_index=False)
        .size()
        .rename(columns={'size': 'documents'})
        .sort_values('documents', ascending=False)
    )
    exported['topic_prevalence'] = save_df(topic_prevalence_df, 'topic_prevalence', folders['tables'])
except Exception as exc:
    notes.append(f'Row-level Top2Vec assignments skipped: {exc}')

# Metric summary
metric_rows = []
if 'cv_top2vec' in globals():
    metric_rows.append({'metric': 'coherence_c_v', 'value': float(cv_top2vec)})
if 'div_top2vec' in globals():
    metric_rows.append({'metric': 'topic_diversity', 'value': float(div_top2vec)})
if metric_rows:
    metrics_df = pd.DataFrame(metric_rows)
    exported['metrics'] = save_df(metrics_df, 'metrics_summary', folders['tables'])

# Model save
model_path = folders['models'] / 'top2vec_model'
try:
    model.save(str(model_path))
    exported['model_file'] = str(model_path)
except Exception as exc:
    notes.append(f'Model save skipped: {exc}')

summary = {
    'pipeline': pipeline_name,
    'run_tag': run_tag,
    'export_root': str(base_dir),
    'saved_items': sorted(exported.keys()),
    'paths': exported,
    'notes': notes
}
summary_path = folders['meta'] / 'export_summary.json'
with open(summary_path, 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2, ensure_ascii=False)

print(f'Export complete: {summary_path}')
print('Saved items:', ', '.join(sorted(exported.keys())) if exported else 'none')
if notes:
    print('Notes:')
    for note in notes:
        print('-', note)

Export complete: Outputs\Comparisons_03\Top2Vec\20260416_213040\meta\export_summary.json
Saved items: metrics, model_file, row_topic_assignments, topic_prevalence, topic_terms
